# GraphSL baselines — dữ liệu đã vá lỗi (Colab GPU)

**Mục đích:** bảng 5 baseline × 3 đồ thị (LPSI, NetSleuth, GCNSI, IVGD, SL-VAE × Karate, Jazz, NetScience) — Review 30/10 Mục 7.4, nộp 10/11.

**Vì sao cần vá:** trong GraphSL 0.15, `diffusion_generation` truyền tập nguồn cho ndlib dưới dạng vector 0/1, trong khi ndlib cần danh sách mã node, nên mọi cascade lan từ node 0 và node 1. Notebook này dùng bản sao đúng từng dòng của hàm đó, **chỉ sửa một chỗ**: truyền danh sách mã node nguồn. Mọi tham số khác giữ mặc định GraphSL (nguồn chọn trong nhóm bậc cao, IC với xác suất `threshold` = 0,5, 10 bước, trung bình 10 lần mô phỏng, chia 60/40 với seed 0).

**Trước khi chạy:** Runtime → Change runtime type → **GPU (T4)**. Nên chạy ô 3 (Google Drive) để kết quả được lưu dần, không mất khi Colab ngắt kết nối. Chạy lần lượt các ô; ô 7 là ô chạy chính.

In [ ]:
# Ô 1: kiểm tra GPU
import torch
print("torch", torch.__version__, "| CUDA:", torch.cuda.is_available(), "|",
      torch.cuda.get_device_name(0) if torch.cuda.is_available() else "KHÔNG CÓ GPU - đổi runtime sang GPU")

In [ ]:
# Ô 2: cài đúng phiên bản như máy cá nhân (torch giữ bản CUDA sẵn có của Colab)
!pip -q install GraphSL==0.15 ndlib==6.0.1 torch-geometric==2.8.1 six
!pip index versions GraphSL 2>/dev/null | head -2   # xem có bản GraphSL mới hơn 0.15 không (để đối chiếu repo)

In [ ]:
# Ô 3 (nên chạy): lưu kết quả vào Google Drive để không mất khi bị ngắt kết nối
import os
try:
    from google.colab import drive
    drive.mount('/content/drive')
    OUT = '/content/drive/MyDrive/GraphSL_colab'
except Exception as e:
    print("Không mount được Drive, lưu tạm trong /content:", e)
    OUT = '/content/GraphSL_colab'
os.makedirs(OUT, exist_ok=True); print("Kết quả lưu tại:", OUT)

In [ ]:
# Ô 4: hàm sinh dữ liệu — bản sao GraphSL 0.15 utils.diffusion_generation, chỉ IC, có cờ fixed
import random, numpy as np, networkx as nx, torch
import ndlib.models.ModelConfig as mc, ndlib.models.epidemics as ep
from GraphSL.utils import generate_seed_vector

def diffusion_generation_v(graph, sim_num=100, time_step=10, repeat_step=10, seed_ratio=0.2,
                           threshold=0.5, random_seed=0, fixed=True):
    """fixed=False: y hệt GraphSL 0.15 (lỗi: ndlib nhận vector 0/1 -> nhiễm node 0 và 1).
       fixed=True : truyền danh sách mã node nguồn (bản vá, sửa đúng 1 chỗ)."""
    adj_mat = graph['adj_mat']
    G = nx.from_scipy_sparse_array(adj_mat)
    node_num = len(G.nodes()); seed_num = int(seed_ratio * node_num)
    degree_list = sorted(G.degree(), key=lambda x: x[1], reverse=True)
    num_more_node = int(len(degree_list) * 0.05)
    top_nodes = [x[0] for x in degree_list[:num_more_node + seed_num]]
    simulation = []
    for i in range(sim_num):
        seed_vector = generate_seed_vector(top_nodes, seed_num, G, random_seed + i)
        init = [v for v, s in zip(G.nodes(), seed_vector) if s == 1] if fixed else seed_vector   # <-- chỗ sửa duy nhất
        inf_vec_all = torch.zeros(node_num)
        config = mc.Configuration()
        for k in range(repeat_step):
            model = ep.IndependentCascadesModel(G, random_seed + k)
            for e in G.edges():
                config.add_edge_configuration("threshold", e, threshold)
            config.add_model_initial_configuration("Infected", init)
            model.set_initial_status(config)
            iterations = model.iteration_bunch(time_step)
            node_status = iterations[0]['status']
            for j in range(1, len(iterations)):
                node_status.update(iterations[j]['status'])
            inf_vec = np.array(list(node_status.values())); inf_vec[inf_vec == 2] = 1
            inf_vec_all += inf_vec
        inf_vec_all = inf_vec_all / repeat_step
        simulation.append([seed_vector, inf_vec_all])
    simulation = torch.Tensor(np.array([[np.asarray(s, dtype=float), v.numpy()] for s, v in simulation])).permute(0, 2, 1)
    return {'adj_mat': adj_mat, 'diff_mat': simulation}

In [ ]:
# Ô 5: tải dữ liệu đồ thị + kiểm tra lỗi và bản vá (vài phút)
import os, datetime as dt
from sklearn.metrics import roc_auc_score
from GraphSL.utils import load_dataset, split_dataset, download_dataset
WORK = '/content/graphsl'; os.makedirs(WORK, exist_ok=True); os.chdir(WORK)
if not os.path.exists('data'): download_dataset(WORK)
LOG = os.path.join(OUT, 'colab_run.log')
def log(m):
    line = f"[{dt.datetime.now():%H:%M:%S}] {m}"; print(line, flush=True); open(LOG, 'a').write(line + "\n")
for ds in ['karate', 'jazz', 'netscience']:
    g = load_dataset(ds, data_dir=WORK)
    for fixed in (False, True):
        D = diffusion_generation_v(g, sim_num=5, fixed=fixed)['diff_mat']
        seeds, diff = D[:, :, 0].numpy(), D[:, :, 1].numpy()
        same = all(np.allclose(diff[0], diff[i]) for i in range(1, 5))
        auc = np.mean([roc_auc_score(seeds[i], diff[i]) for i in range(5)])
        log(f"KIỂM TRA {ds:10s} {'vá ' if fixed else 'gốc'}: tỉ lệ nhiễm TB={diff.mean():.3f} | "
            f"vector nhiễm giống hệt giữa cascade: {same} | AUC(vector nhiễm vs nguồn)={auc:.3f}")
# Kỳ vọng: bản 'gốc' -> giống hệt = True; bản 'vá' -> False.
# Lưu ý: với mặc định GraphSL (20% nguồn, p = 0,5, 10 bước) tập nhiễm gần bão hòa (Jazz ~98%),
# nên AUC(vector nhiễm vs nguồn) chỉ hơi trên 0,5 — đó là do thiết lập, không phải lỗi.
# Thử trên cloud (Claude): Karate gốc 0,810 -> vá 0,871; Jazz 0,547 -> 0,543; vector khác nhau sau khi vá.

In [ ]:
# Ô 6: hàm chạy 5 baseline + GCNSI các chế độ chọn tập nguồn (top-k với k thật; ngưỡng chọn trên val)
import csv, time, traceback
from GraphSL.Prescribed import LPSI, NetSleuth
from GraphSL.GNN.GCNSI.main import GCNSI
from GraphSL.GNN.IVGD.main import IVGD
from GraphSL.GNN.SLVAE.main import SLVAE
from scipy.sparse import csgraph, coo_matrix
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score
CSV = os.path.join(OUT, 'baselines_colab.csv')
FIELDS = ['time','data','dataset','n_nodes','method','seed','acc','pr','re','f1','auc','train_s','test_s','status']
DEV = torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'cpu'

def done(data, ds, mth):
    if not os.path.exists(CSV): return False
    return any(r['data']==data and r['dataset']==ds and r['method']==mth and r['status'].startswith('ok') for r in csv.DictReader(open(CSV)))

def write(row):
    new = not os.path.exists(CSV)
    with open(CSV, 'a', newline='') as f:
        w = csv.DictWriter(f, fieldnames=FIELDS, extrasaction='ignore'); new and w.writeheader(); w.writerow(row)

def set_seed(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s)

def run_method(name, adj, train, test):
    t0 = time.time()
    if name == 'LPSI':
        m = LPSI(); alpha, thres, *_ = m.train(adj, train); t1 = time.time(); r = m.test(adj, test, alpha, thres)
    elif name == 'NetSleuth':
        m = NetSleuth(); k, *_ = m.train(adj, train); t1 = time.time(); r = m.test(adj, test, k)
    elif name == 'GCNSI':
        m = GCNSI(); model, thres, *_ = m.train(adj, train); t1 = time.time(); r = m.test(adj, test, model, thres)
    elif name == 'IVGD':
        m = IVGD(); diff = m.train_diffusion(adj, train); model, thres, *_ = m.train(adj, train, diff)
        t1 = time.time(); r = m.test(adj, test, diff, model, thres)
    elif name == 'SLVAE':
        m = SLVAE(); model, seed_train, thres, *_ = m.train(adj, train); t1 = time.time(); r = m.infer(test, model, seed_train, thres)
    return [r.acc, r.pr, r.re, r.f1, r.auc], t1 - t0, time.time() - t1

# --- GCNSI: đánh giá lại với đầu vào đúng từng cascade (GraphSL.test dùng nhầm lpsi_input[i] cố định) ---
def gcnsi_feats(adj, dataset, alpha=0.01):
    S = torch.tensor(coo_matrix.todense(csgraph.laplacian(adj, normed=True)), dtype=torch.float)
    n = adj.shape[0]; M = (1 - alpha) * torch.linalg.inv(torch.eye(n) - alpha * S)
    out = []
    for im in dataset:
        d = im[:, -1].float(); V3 = d.clone(); V4 = d.clone(); V3[d < 0.5] = 0.5; V4[d >= 0.5] = 0.5
        out.append(torch.cat([d.unsqueeze(1)] + [(M @ v).unsqueeze(1) for v in (d, V3, V4)], dim=1))
    return out

def gcnsi_scores(model, adj, dataset):
    coo = adj.tocoo(); ei = torch.stack([torch.from_numpy(coo.row.astype(np.int64)), torch.from_numpy(coo.col.astype(np.int64))])
    model = model.cpu().eval()
    with torch.no_grad():
        return [torch.softmax(model(x, ei), dim=1)[:, 1].numpy() for x in gcnsi_feats(adj, dataset)]

def evaluate(dataset, preds, rule):
    m = np.zeros(5)
    for im, p in zip(dataset, preds):
        y = im[:, 0].numpy().astype(int); yh = rule(p, y)
        m += [accuracy_score(y, yh), precision_score(y, yh, zero_division=1), recall_score(y, yh, zero_division=1), f1_score(y, yh, zero_division=1), roc_auc_score(y, p)]
    return list(m / len(preds))

def topk(p, y):
    yh = np.zeros_like(y); yh[np.argsort(-p)[:int(y.sum())]] = 1; return yh

def best_thres(dataset, preds, num=10):
    a = np.concatenate(preds); best = (-1, None)
    for t in np.linspace(a.min(), a.max(), num + 2)[1:-1]:
        f = np.mean([f1_score(im[:, 0].numpy().astype(int), p >= t) for im, p in zip(dataset, preds)])
        if f > best[0]: best = (f, float(t))
    return best[1]

def gcnsi_modes(adj, train, test):
    set_seed(0); g = GCNSI(); model, thres, *_ = g.train(adj, train)
    P = gcnsi_scores(model, adj, test)
    res = {'GCNSI_sua_nguong_train': evaluate(test, P, lambda p, y: (p >= thres).astype(int)),
           'GCNSI_topk_k_that': evaluate(test, P, topk)}
    idx = np.random.default_rng(0).permutation(len(train)); cut = int(0.75 * len(train))
    tr = torch.utils.data.Subset(train, idx[:cut].tolist()); va = torch.utils.data.Subset(train, idx[cut:].tolist())
    set_seed(0); mv, *_ = GCNSI().train(adj, tr); tv = best_thres(va, gcnsi_scores(mv, adj, va))
    res['GCNSI_nguong_val'] = evaluate(test, gcnsi_scores(mv, adj, test), lambda p, y: (p >= tv).astype(int))
    return res

def run_all(data, datasets, methods, modes=True):
    for ds in datasets:
        set_seed(0)
        adj, train, test = split_dataset(diffusion_generation_v(load_dataset(ds, data_dir=WORK), fixed=(data == 'va')))
        log(f"== [{data}] {ds} (n={adj.shape[0]}): {len(train)} train / {len(test)} test")
        for mth in methods:
            if done(data, ds, mth): log(f"   {mth}: đã có, bỏ qua"); continue
            set_seed(0); row = dict(time=dt.datetime.now().isoformat(timespec='seconds'), data=data, dataset=ds,
                                    n_nodes=adj.shape[0], method=mth, seed=0)
            try:
                log(f"   {mth}: đang huấn luyện..."); m, tr_s, te_s = run_method(mth, adj, train, test)
                row.update(dict(zip(['acc','pr','re','f1','auc'], m)), train_s=round(tr_s,1), test_s=round(te_s,1), status=f'ok (colab {DEV})')
                log(f"   {mth}: acc={m[0]:.3f} pr={m[1]:.3f} re={m[2]:.3f} f1={m[3]:.3f} auc={m[4]:.3f} | train {tr_s:.0f}s, test {te_s:.0f}s")
            except Exception as e:
                row.update(status=f'error: {type(e).__name__}: {e}'); log(f"   {mth}: LỖI {type(e).__name__}: {e}")
                open(LOG, 'a').write(traceback.format_exc() + "\n")
            write(row)
        if modes and not done(data, ds, 'GCNSI_topk_k_that'):
            try:
                t0 = time.time(); log("   GCNSI các chế độ: đang chạy...")
                for mode, m in gcnsi_modes(adj, train, test).items():
                    write(dict(time=dt.datetime.now().isoformat(timespec='seconds'), data=data, dataset=ds, n_nodes=adj.shape[0],
                               method=mode, seed=0, acc=m[0], pr=m[1], re=m[2], f1=m[3], auc=m[4], status=f'ok (colab {DEV})'))
                    log(f"   {mode}: acc={m[0]:.3f} pr={m[1]:.3f} re={m[2]:.3f} f1={m[3]:.3f} auc={m[4]:.3f}")
                log(f"   GCNSI các chế độ: xong {time.time()-t0:.0f}s")
            except Exception as e:
                log(f"   GCNSI các chế độ: LỖI {type(e).__name__}: {e}"); open(LOG, 'a').write(traceback.format_exc() + "\n")

**Ô 7 — chạy chính trên dữ liệu đã vá.** Thứ tự: Karate → Jazz → NetScience, mỗi đồ thị 5 phương pháp rồi GCNSI các chế độ. Nếu Colab ngắt kết nối: chạy lại ô 1–6 rồi ô 7; những phần đã có trong CSV (trên Drive) sẽ được bỏ qua.

In [ ]:
run_all('va', ['karate', 'jazz', 'netscience'], ['LPSI', 'NetSleuth', 'GCNSI', 'IVGD', 'SLVAE'])

**Ô 8 (tùy chọn) — hoàn tất bảng mặc định GraphSL (dữ liệu lỗi) trên NetScience** cho các ô máy cá nhân chưa chạy kịp. Chỉ dùng làm bằng chứng lỗi, không phải kết quả baseline. Có thể bỏ qua nếu hết thời gian.

In [ ]:
run_all('goc', ['netscience'], ['GCNSI', 'IVGD', 'SLVAE'], modes=False)

In [ ]:
# Ô 9: xem bảng + ghi phiên bản + tải về
import pandas as pd
df = pd.read_csv(CSV); print(df[['data','dataset','method','acc','pr','re','f1','auc','train_s','status']].to_string(index=False))
!pip freeze > {OUT}/requirements_colab.txt
from google.colab import files
for fn in ['baselines_colab.csv', 'colab_run.log', 'requirements_colab.txt']:
    files.download(os.path.join(OUT, fn))

Sau khi tải về: chép 3 file vào `C:\\Users\\ad\\Documents\\GraphSL_baselines\\results\\colab\\` rồi báo Claude để ghép bảng và cập nhật báo cáo.